# Pratilipi Assignment: Book Recommendation System
This notebook outlines a hybrid recommendation system. 
Since chapters are ordered, we need to recommend:
1. The **Next Chapter** for books the user is currently reading.
2. A **New Book** (Chapter 1) if the user needs something new.

*Requirement constraint: We will not use pre-trained models or external deep ML libraries like `scikit-learn`. TF-IDF and Cosine Similarity are implemented entirely from scratch utilizing raw python collections.*\n

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')\n

## 1. Data Loading and Preparation
First, we'll load the datasets and take a look at the shape of our data.\n

In [ ]:
print("Loading data...")
chapters_df = pd.read_csv('chapters.csv')
interactions_df = pd.read_csv('interactions.csv')

print(f"Chapters shape: {chapters_df.shape}")
print(f"Interactions shape: {interactions_df.shape}")\n

We need to aggregate the chapter data up to the book level to do book-level recommendations. We'll join all tags for a book together so we have metadata characterizing each book.\n

In [ ]:
# Fill missing/null tags
chapters_df['tags'] = chapters_df['tags'].fillna('')

# Group chapters up to the book level
book_metadata = chapters_df.groupby('book_id').agg({
    'author_id': 'first', 
    'tags': lambda x: ' '.join(set('|'.join(x).replace('|', ' ').split())),
    'chapter_sequence_no': 'max'
}).reset_index()

book_metadata.rename(columns={'chapter_sequence_no': 'total_chapters'}, inplace=True)
display(book_metadata.head())\n

## 2. 'Continue Reading' Recommender
If a user has read some chapters of a book, the most natural recommendation is simply the *next* chapter in the sequence. 
This is purely deterministic logic.\n

In [ ]:
# Dictionary holding the max chapter sequence available for each book
book_max_chapters = chapters_df.groupby('book_id')['chapter_sequence_no'].max().to_dict()

def recommend_next_chapters(user_id, interactions, top_k=None):
    # Get the user's reading history
    user_history = interactions[interactions['user_id'] == user_id]
    
    # Find the maximum chapter sequence they have read for each book
    user_progress = user_history.groupby('book_id')['chapter_sequence_no'].max()
    
    recommendations = {}
    for book_id, max_read_chap in user_progress.items():
        max_available = book_max_chapters.get(book_id, max_read_chap)
        # If there are more chapters left, recommend the next one
        if max_read_chap < max_available:
            recommendations[book_id] = max_read_chap + 1
            
    return recommendations\n

## 3. 'New Book' Recommender (TF-IDF from Scratch)
When a user is looking for a new book, we want to recommend books similar to what they've read before based on metadata (tags and author).
Rather than utilizing `sklearn.feature_extraction.text.TfidfVectorizer`, we hand-code a memory-bound TF-IDF vectorizer and a sparse Cosine Similarity distance calculator.\n

In [ ]:
class CustomTFIDF:
    def __init__(self):
        self.doc_freqs = defaultdict(int)
        self.idf = {}
        self.num_docs = 0
        
    def fit_transform(self, text_corpus):
        self.num_docs = len(text_corpus)
        
        # Calculate Document Frequency (DF)
        for doc in text_corpus:
            words = set(doc.split())
            for word in words:
                self.doc_freqs[word] += 1
                
        # Calculate Inverse Document Frequency (IDF)
        for word, df in self.doc_freqs.items():
            self.idf[word] = math.log((1 + self.num_docs) / (1 + df)) + 1
            
        # Create normalized TF-IDF vectors (stored as sparse dictionaries)
        vectors = []
        for doc in text_corpus:
            counts = Counter(doc.split())
            vec = {}
            norm = 0.0
            
            for word, freq in counts.items():
                weight = freq * self.idf[word]
                vec[word] = weight
                norm += weight ** 2
            
            # Mathematical L2 Normalization
            norm = math.sqrt(norm)
            if norm > 0:
                for word in vec:
                    vec[word] /= norm
            vectors.append(vec)
            
        return vectors

def sparse_cosine_similarity(vec1, vec2):
    # Iterate over the smaller dict for computational speed
    if len(vec1) > len(vec2):
        vec1, vec2 = vec2, vec1
        
    score = 0.0
    for word, weight in vec1.items():
        if word in vec2:
            score += weight * vec2[word]
    return score

def get_user_profile(user_read_indices, book_vectors):
    # A user's geometric profile merges (averages) the TF-IDF components of everything they've previously read.
    profile = defaultdict(float)
    n = len(user_read_indices)
    if n == 0: return dict(profile)
    
    for idx in user_read_indices:
        for word, weight in book_vectors[idx].items():
            profile[word] += weight / n
            
    return dict(profile)\n

Now, let's process the book features and generate the core recommendation driver.\n

In [ ]:
print("Generating TF-IDF matrices from scratch...")
# Combine tags and author to form the unique 'document' mapped feature extraction pipeline for each book
text_documents = book_metadata['tags'] + " author_" + book_metadata['author_id'].astype(str)
text_corpus = [str(text) for text in text_documents]

tfidf_model = CustomTFIDF()
book_vectors = tfidf_model.fit_transform(text_corpus)
book_ids_array = book_metadata['book_id'].values

# Popularity baseline for cold-start fallback
popular_books = interactions_df['book_id'].value_counts().index.tolist()

def recommend_new_books(user_id, interactions_subset, top_k=5):
    user_history = interactions_subset[interactions_subset['user_id'] == user_id]['book_id'].unique()
    
    # Cold start: Return popular books if no history
    if len(user_history) == 0:
        return popular_books[:top_k]
        
    # Find coordinates of history locally
    read_indices_set = set(book_metadata.index[book_metadata['book_id'].isin(user_history)])
    read_indices = list(read_indices_set)
    
    if not read_indices:
         return popular_books[:top_k]
         
    # Formulate User Taste embedding projection
    user_profile = get_user_profile(read_indices, book_vectors)
    
    # Calculate similarity angles against 50,000+ localized documents (excluding already read items)
    sim_scores = np.zeros(len(book_vectors))
    for i, b_vec in enumerate(book_vectors):
        if i in read_indices_set:
            sim_scores[i] = -1.0 
        else:
            sim_scores[i] = sparse_cosine_similarity(user_profile, b_vec)
            
    # NumPy optimization for retrieving unordered top results extremely fast
    top_idx = np.argpartition(sim_scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(-sim_scores[top_idx])]
    
    return book_ids_array[top_idx].tolist()\n

## 4. Evaluation 
Let's validate the Recommender using an offline Leave-One-Out cross validation method. We'll hold out a random book per user and observe prediction reliability.\n

In [ ]:
# Bind features together to calculate valid history overlaps
merged_df = pd.merge(
    interactions_df, 
    chapters_df[['chapter_id', 'chapter_sequence_no']], 
    on='chapter_id', 
    how='left'
)

def evaluate_recommender(num_eval_users=500, top_k=10):
    book_counts = merged_df.groupby('user_id')['book_id'].nunique()
    valid_users = book_counts[book_counts > 1].index
    
    np.random.seed(42)
    sample_users = np.random.choice(valid_users, num_eval_users, replace=False)
    
    hits = 0
    total = 0
    
    print(f"Evaluating Hit Rate @ {top_k} on {num_eval_users} sample users...")
    
    for i, u_id in enumerate(sample_users):
        user_books = merged_df[merged_df['user_id'] == u_id]['book_id'].unique().tolist()
        
        # Sequester one ground-truth truth book
        hidden_book = np.random.choice(user_books)
        training_books = [b for b in user_books if b != hidden_book]
        
        fake_interactions = pd.DataFrame({'user_id': [u_id]*len(training_books), 'book_id': training_books})
        preds = recommend_new_books(u_id, fake_interactions, top_k=top_k)
        
        if hidden_book in preds:
            hits += 1
        total += 1
        
        if (i+1) % 100 == 0:
            print(f"  Handled {i+1} users...")

    score = hits / total if total > 0 else 0
    print(f"Final Validation HR@{top_k}: {score:.4f}")

evaluate_recommender(num_eval_users=200, top_k=10)\n

## 5. Demonstration Pipeline
Lastly, lets push a single user visually entirely through our pipeline algorithms.\n

In [ ]:
demo_user = interactions_df['user_id'].iloc[0]
user_interacted_books = interactions_df[interactions_df['user_id'] == demo_user]['book_id'].unique().tolist()

print(f"User '{demo_user}' previously interacted with book IDs: {user_interacted_books}")

cr_recs = recommend_next_chapters(demo_user, merged_df)
print("
[Continue Reading Output] ==>")
for book, nxt_chap in cr_recs.items():
    print(f" Continue Book {book} at Chapter Sequence: {nxt_chap}")

nb_recs = recommend_new_books(demo_user, merged_df, top_k=3)
print("
[Start a New Book Output] ==>")
print(f" Top 3 Algorithmic Book Suggestions: {nb_recs}")\n